# Model Diagnostics Deep Dive\n\n**Chapter 6: Regression Techniques for Soccer Analytics - EXTRA**\n\n## Learning Objectives\n\n- Perform comprehensive residual analysis\n- Detect and handle heteroscedasticity\n- Identify influential observations and outliers\n- Test regression assumptions rigorously\n- Use diagnostic plots effectively\n- Apply remedial measures when assumptions are violated\n- Understand leverage, influence, and Cook's distance

In [ ]:
import pandas as pd\nimport numpy as np\nimport matplotlib.pyplot as plt\nimport seaborn as sns\nimport statsmodels.api as sm\nimport statsmodels.formula.api as smf\nfrom statsmodels.stats.diagnostic import het_breuschpagan, het_white\nfrom statsmodels.stats.outliers_influence import variance_inflation_factor\nfrom scipy import stats\nfrom sklearn.linear_model import LinearRegression\nfrom sklearn.preprocessing import StandardScaler\n\nsns.set_style('whitegrid')\nplt.rcParams['figure.figsize'] = (14, 10)

## Regression Assumptions\n\nLinear regression makes several key assumptions:\n\n1. **Linearity:** Relationship between X and y is linear\n2. **Independence:** Observations are independent\n3. **Homoscedasticity:** Constant variance of residuals\n4. **Normality:** Residuals are normally distributed\n5. **No multicollinearity:** Features are not highly correlated\n\n**Why check?** Violations can lead to:\n- Biased coefficient estimates\n- Incorrect standard errors\n- Invalid hypothesis tests\n- Poor predictions

## Load Data and Fit Model

In [ ]:
# Generate realistic match data\nnp.random.seed(42)\nn = 200\n\ndf = pd.DataFrame({\n    'shots_on_target': np.random.randint(3, 15, n),\n    'possession': np.random.uniform(35, 70, n),\n    'xg': np.random.uniform(0.5, 3.5, n),\n    'opponent_strength': np.random.uniform(0.3, 0.9, n),\n    'home': np.random.choice([0, 1], n)\n})\n\n# Generate goals with some heteroscedasticity\ndf['goals'] = (\n    df['xg'] * 0.9 + \n    df['shots_on_target'] * 0.08 + \n    df['home'] * 0.3 - \n    df['opponent_strength'] * 0.4 +\n    np.random.normal(0, 0.3 + df['xg'] * 0.1, n)  # Variance increases with xG\n).clip(0, 6).round().astype(int)\n\nprint(\"Dataset loaded:\")\nprint(df.head())\nprint(f\"\\nShape: {df.shape}\")\n\n# Fit model using statsmodels (provides more diagnostics)\nformula = 'goals ~ shots_on_target + possession + xg + opponent_strength + home'\nmodel = smf.ols(formula, data=df).fit()\n\nprint(f\"\\nModel Summary:\")\nprint(model.summary())

## 1. Residual Analysis\n\n**Residuals** = Actual - Predicted\n\n**What to look for:**\n- Random scatter (no patterns)\n- Centered around zero\n- Constant spread (homoscedasticity)

In [ ]:
# Get residuals and fitted values\nresiduals = model.resid\nfitted_values = model.fittedvalues\nstandardized_resid = model.resid_pearson\n\n# Create comprehensive residual plots\nfig, axes = plt.subplots(2, 3, figsize=(16, 10))\n\n# 1. Residuals vs Fitted\naxes[0, 0].scatter(fitted_values, residuals, alpha=0.6)\naxes[0, 0].axhline(y=0, color='r', linestyle='--', linewidth=2)\naxes[0, 0].set_xlabel('Fitted Values')\naxes[0, 0].set_ylabel('Residuals')\naxes[0, 0].set_title('Residuals vs Fitted')\naxes[0, 0].grid(True, alpha=0.3)\n\n# 2. Scale-Location (sqrt of standardized residuals)\naxes[0, 1].scatter(fitted_values, np.sqrt(np.abs(standardized_resid)), alpha=0.6)\naxes[0, 1].set_xlabel('Fitted Values')\naxes[0, 1].set_ylabel('√|Standardized Residuals|')\naxes[0, 1].set_title('Scale-Location Plot')\naxes[0, 1].grid(True, alpha=0.3)\n\n# 3. Normal Q-Q Plot\nstats.probplot(residuals, dist=\"norm\", plot=axes[0, 2])\naxes[0, 2].set_title('Normal Q-Q Plot')\n\n# 4. Histogram of residuals\naxes[1, 0].hist(residuals, bins=20, edgecolor='black', alpha=0.7)\naxes[1, 0].axvline(x=0, color='r', linestyle='--', linewidth=2)\naxes[1, 0].set_xlabel('Residuals')\naxes[1, 0].set_ylabel('Frequency')\naxes[1, 0].set_title('Distribution of Residuals')\n\n# 5. Residuals vs each predictor (example: xG)\naxes[1, 1].scatter(df['xg'], residuals, alpha=0.6)\naxes[1, 1].axhline(y=0, color='r', linestyle='--', linewidth=2)\naxes[1, 1].set_xlabel('xG')\naxes[1, 1].set_ylabel('Residuals')\naxes[1, 1].set_title('Residuals vs xG')\naxes[1, 1].grid(True, alpha=0.3)\n\n# 6. Residuals vs Order (time series check)\naxes[1, 2].scatter(range(len(residuals)), residuals, alpha=0.6)\naxes[1, 2].axhline(y=0, color='r', linestyle='--', linewidth=2)\naxes[1, 2].set_xlabel('Observation Order')\naxes[1, 2].set_ylabel('Residuals')\naxes[1, 2].set_title('Residuals vs Order')\naxes[1, 2].grid(True, alpha=0.3)\n\nplt.tight_layout()\nplt.show()\n\nprint(\"Diagnostic Checklist:\")\nprint(\"✓ Residuals vs Fitted: Should show random scatter, no pattern\")\nprint(\"✓ Scale-Location: Should be flat (constant variance)\")\nprint(\"✓ Q-Q Plot: Points should follow diagonal line (normality)\")\nprint(\"✓ Histogram: Should be bell-shaped and centered at 0\")\nprint(\"✓ Residuals vs Predictors: No patterns\")\nprint(\"✓ Residuals vs Order: No trends (independence)\")

## 2. Test for Heteroscedasticity\n\n**Heteroscedasticity:** Non-constant variance of residuals\n\n**Tests:**\n- **Breusch-Pagan Test:** Tests if variance depends on predictors\n- **White Test:** More general test for heteroscedasticity

In [ ]:
# Breusch-Pagan test\nX = df[['shots_on_target', 'possession', 'xg', 'opponent_strength', 'home']]\nX_with_const = sm.add_constant(X)\n\nbp_test = het_breuschpagan(residuals, X_with_const)\nbp_statistic, bp_pvalue, bp_f_stat, bp_f_pvalue = bp_test\n\nprint(\"Breusch-Pagan Test for Heteroscedasticity:\")\nprint(f\"  LM Statistic: {bp_statistic:.3f}\")\nprint(f\"  p-value: {bp_pvalue:.4f}\")\nprint(f\"\\nInterpretation:\")\nif bp_pvalue < 0.05:\n    print(\"  ⚠️ Reject null hypothesis - Heteroscedasticity detected!\")\n    print(\"  Remedies:\")\n    print(\"    - Transform dependent variable (log, sqrt)\")\n    print(\"    - Use weighted least squares\")\n    print(\"    - Use robust standard errors\")\nelse:\n    print(\"  ✅ Fail to reject null - Homoscedasticity assumption holds\")\n\n# White test\nwhite_test = het_white(residuals, X_with_const)\nwhite_statistic, white_pvalue, white_f_stat, white_f_pvalue = white_test\n\nprint(f\"\\nWhite Test for Heteroscedasticity:\")\nprint(f\"  LM Statistic: {white_statistic:.3f}\")\nprint(f\"  p-value: {white_pvalue:.4f}\")\nif white_pvalue < 0.05:\n    print(\"  ⚠️ Heteroscedasticity detected by White test\")\nelse:\n    print(\"  ✅ No heteroscedasticity detected\")

## 3. Influential Observations\n\n**Key concepts:**\n- **Leverage:** How far a point's X values are from the mean\n- **Influence:** How much a point affects the regression line\n- **Cook's Distance:** Combined measure of leverage and influence\n\n**Thresholds:**\n- Cook's D > 4/n: Potentially influential\n- Cook's D > 1: Highly influential

In [ ]:
# Get influence measures\ninfluence = model.get_influence()\nleverage = influence.hat_matrix_diag\ncooks_d = influence.cooks_distance[0]\ndfbetas = influence.dfbetas\n\n# Identify influential points\nthreshold = 4 / len(df)\ninfluential_idx = np.where(cooks_d > threshold)[0]\n\nprint(f\"Influential Observations Analysis:\")\nprint(f\"  Threshold (4/n): {threshold:.4f}\")\nprint(f\"  Number of influential points: {len(influential_idx)}\")\nprint(f\"  Percentage: {len(influential_idx)/len(df)*100:.1f}%\")\n\nif len(influential_idx) > 0:\n    print(f\"\\nTop 5 most influential observations:\")\n    top_influential = np.argsort(cooks_d)[-5:][::-1]\n    for i, idx in enumerate(top_influential, 1):\n        print(f\"  {i}. Index {idx}: Cook's D = {cooks_d[idx]:.4f}\")\n        print(f\"     Goals: {df.iloc[idx]['goals']}, xG: {df.iloc[idx]['xg']:.2f}, Predicted: {fitted_values[idx]:.2f}\")\n\n# Visualize influence\nfig, axes = plt.subplots(1, 3, figsize=(16, 5))\n\n# 1. Cook's Distance\naxes[0].stem(range(len(cooks_d)), cooks_d, markerfmt=',')\naxes[0].axhline(y=threshold, color='r', linestyle='--', label=f'Threshold ({threshold:.4f})')\naxes[0].set_xlabel('Observation Index')\naxes[0].set_ylabel(\"Cook's Distance\")\naxes[0].set_title(\"Cook's Distance Plot\")\naxes[0].legend()\naxes[0].grid(True, alpha=0.3)\n\n# 2. Leverage vs Residuals\naxes[1].scatter(leverage, standardized_resid, alpha=0.6)\naxes[1].axhline(y=0, color='r', linestyle='--')\naxes[1].axhline(y=2, color='orange', linestyle='--', alpha=0.5)\naxes[1].axhline(y=-2, color='orange', linestyle='--', alpha=0.5)\naxes[1].set_xlabel('Leverage')\naxes[1].set_ylabel('Standardized Residuals')\naxes[1].set_title('Leverage vs Standardized Residuals')\naxes[1].grid(True, alpha=0.3)\n\n# Highlight influential points\nif len(influential_idx) > 0:\n    axes[1].scatter(leverage[influential_idx], standardized_resid[influential_idx], \n                   color='red', s=100, marker='x', linewidths=3, label='Influential')\n    axes[1].legend()\n\n# 3. Residuals vs Leverage\naxes[2].scatter(leverage, residuals, alpha=0.6)\naxes[2].axhline(y=0, color='r', linestyle='--')\naxes[2].set_xlabel('Leverage')\naxes[2].set_ylabel('Residuals')\naxes[2].set_title('Residuals vs Leverage')\naxes[2].grid(True, alpha=0.3)\n\nplt.tight_layout()\nplt.show()

## 4. Multicollinearity\n\n**Problem:** Highly correlated predictors make coefficients unstable.\n\n**Detection:** Variance Inflation Factor (VIF)\n- VIF = 1: No correlation\n- VIF < 5: Acceptable\n- VIF > 10: Problematic multicollinearity

In [ ]:
# Calculate VIF for each feature\nvif_data = pd.DataFrame()\nvif_data[\"Feature\"] = X.columns\nvif_data[\"VIF\"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]\nvif_data = vif_data.sort_values('VIF', ascending=False)\n\nprint(\"Variance Inflation Factor (VIF):\")\nprint(vif_data.to_string(index=False))\nprint(f\"\\nInterpretation:\")\nprint(f\"  VIF < 5: No multicollinearity concern\")\nprint(f\"  5 ≤ VIF < 10: Moderate multicollinearity\")\nprint(f\"  VIF ≥ 10: Severe multicollinearity\")\n\n# Check for problematic VIF\nproblematic = vif_data[vif_data['VIF'] > 10]\nif len(problematic) > 0:\n    print(f\"\\n⚠️ Features with severe multicollinearity:\")\n    for _, row in problematic.iterrows():\n        print(f\"  - {row['Feature']}: VIF = {row['VIF']:.2f}\")\n    print(f\"\\nRemedies:\")\n    print(f\"  - Remove one of the correlated features\")\n    print(f\"  - Combine correlated features (e.g., PCA)\")\n    print(f\"  - Use regularization (Ridge regression)\")\nelse:\n    print(f\"\\n✅ No severe multicollinearity detected\")\n\n# Visualize correlation matrix\nplt.figure(figsize=(10, 8))\ncorr_matrix = X.corr()\nsns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, \n            square=True, linewidths=1, cbar_kws={\"shrink\": 0.8})\nplt.title('Feature Correlation Matrix')\nplt.tight_layout()\nplt.show()

## 5. Normality Tests\n\n**Tests for normality of residuals:**\n- **Shapiro-Wilk Test:** Most powerful for small samples\n- **Jarque-Bera Test:** Based on skewness and kurtosis\n- **Anderson-Darling Test:** More sensitive to tails

In [ ]:
# Shapiro-Wilk test\nshapiro_stat, shapiro_pvalue = stats.shapiro(residuals)\n\nprint(\"Normality Tests:\")\nprint(f\"\\nShapiro-Wilk Test:\")\nprint(f\"  Statistic: {shapiro_stat:.4f}\")\nprint(f\"  p-value: {shapiro_pvalue:.4f}\")\nif shapiro_pvalue < 0.05:\n    print(\"  ⚠️ Residuals are not normally distributed\")\nelse:\n    print(\"  ✅ Residuals are approximately normal\")\n\n# Jarque-Bera test (included in statsmodels summary)\njb_stat = model.jarque_bera[0]\njb_pvalue = model.jarque_bera[1]\n\nprint(f\"\\nJarque-Bera Test:\")\nprint(f\"  Statistic: {jb_stat:.4f}\")\nprint(f\"  p-value: {jb_pvalue:.4f}\")\nif jb_pvalue < 0.05:\n    print(\"  ⚠️ Residuals are not normally distributed\")\nelse:\n    print(\"  ✅ Residuals are approximately normal\")\n\n# Anderson-Darling test\nanderson_result = stats.anderson(residuals, dist='norm')\n\nprint(f\"\\nAnderson-Darling Test:\")\nprint(f\"  Statistic: {anderson_result.statistic:.4f}\")\nprint(f\"  Critical values: {anderson_result.critical_values}\")\nprint(f\"  Significance levels: {anderson_result.significance_level}%\")\nif anderson_result.statistic > anderson_result.critical_values[2]:  # 5% level\n    print(\"  ⚠️ Residuals are not normally distributed (5% level)\")\nelse:\n    print(\"  ✅ Residuals are approximately normal (5% level)\")\n\nprint(f\"\\nNote: For large samples, slight deviations from normality are less concerning.\")\nprint(f\"Visual inspection (Q-Q plot) is often more informative than tests.\")

## 6. Remedial Measures\n\n### When Assumptions Are Violated

In [ ]:
# Example: Transform dependent variable to address heteroscedasticity\ndf['log_goals_plus_1'] = np.log(df['goals'] + 1)  # +1 to handle zeros\n\n# Fit model with transformed target\nformula_transformed = 'log_goals_plus_1 ~ shots_on_target + possession + xg + opponent_strength + home'\nmodel_transformed = smf.ols(formula_transformed, data=df).fit()\n\nprint(\"Model with Log-Transformed Target:\")\nprint(f\"R²: {model_transformed.rsquared:.3f}\")\nprint(f\"\\nBreusch-Pagan test on transformed model:\")\n\nresiduals_transformed = model_transformed.resid\nbp_test_transformed = het_breuschpagan(residuals_transformed, X_with_const)\nprint(f\"  p-value: {bp_test_transformed[1]:.4f}\")\n\nif bp_test_transformed[1] > 0.05:\n    print(\"  ✅ Transformation helped reduce heteroscedasticity!\")\nelse:\n    print(\"  ⚠️ Heteroscedasticity still present\")\n\n# Compare residual plots\nfig, axes = plt.subplots(1, 2, figsize=(14, 5))\n\naxes[0].scatter(fitted_values, residuals, alpha=0.6)\naxes[0].axhline(y=0, color='r', linestyle='--')\naxes[0].set_xlabel('Fitted Values')\naxes[0].set_ylabel('Residuals')\naxes[0].set_title('Original Model')\naxes[0].grid(True, alpha=0.3)\n\nfitted_transformed = model_transformed.fittedvalues\naxes[1].scatter(fitted_transformed, residuals_transformed, alpha=0.6, color='green')\naxes[1].axhline(y=0, color='r', linestyle='--')\naxes[1].set_xlabel('Fitted Values')\naxes[1].set_ylabel('Residuals')\naxes[1].set_title('Log-Transformed Model')\naxes[1].grid(True, alpha=0.3)\n\nplt.tight_layout()\nplt.show()\n\nprint(\"\\nOther remedial measures:\")\nprint(\"  - Robust standard errors (HC1, HC3)\")\nprint(\"  - Weighted least squares\")\nprint(\"  - Generalized least squares\")\nprint(\"  - Non-parametric methods (if normality severely violated)\")

## Comprehensive Diagnostic Report

In [ ]:
def diagnostic_report(model, X, df):\n    \"\"\"Generate comprehensive diagnostic report\"\"\"\n    print(\"="*60)\n    print(\"COMPREHENSIVE MODEL DIAGNOSTIC REPORT\")\n    print(\"="*60)\n    \n    # 1. Model fit\n    print(f\"\\n1. MODEL FIT\")\n    print(f\"   R²: {model.rsquared:.3f}\")\n    print(f\"   Adjusted R²: {model.rsquared_adj:.3f}\")\n    print(f\"   AIC: {model.aic:.2f}\")\n    print(f\"   BIC: {model.bic:.2f}\")\n    \n    # 2. Residual analysis\n    residuals = model.resid\n    print(f\"\\n2. RESIDUAL ANALYSIS\")\n    print(f\"   Mean: {residuals.mean():.6f} (should be ~0)\")\n    print(f\"   Std Dev: {residuals.std():.3f}\")\n    print(f\"   Min: {residuals.min():.3f}\")\n    print(f\"   Max: {residuals.max():.3f}\")\n    \n    # 3. Normality\n    _, shapiro_p = stats.shapiro(residuals)\n    print(f\"\\n3. NORMALITY TEST\")\n    print(f\"   Shapiro-Wilk p-value: {shapiro_p:.4f}\")\n    print(f\"   {'✅ Normal' if shapiro_p > 0.05 else '⚠️ Not normal'}\")\n    \n    # 4. Heteroscedasticity\n    X_const = sm.add_constant(X)\n    _, bp_p, _, _ = het_breuschpagan(residuals, X_const)\n    print(f\"\\n4. HETEROSCEDASTICITY TEST\")\n    print(f\"   Breusch-Pagan p-value: {bp_p:.4f}\")\n    print(f\"   {'✅ Homoscedastic' if bp_p > 0.05 else '⚠️ Heteroscedastic'}\")\n    \n    # 5. Multicollinearity\n    vif_values = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]\n    max_vif = max(vif_values)\n    print(f\"\\n5. MULTICOLLINEARITY\")\n    print(f\"   Max VIF: {max_vif:.2f}\")\n    print(f\"   {'✅ No concern' if max_vif < 10 else '⚠️ Multicollinearity detected'}\")\n    \n    # 6. Influential observations\n    influence = model.get_influence()\n    cooks_d = influence.cooks_distance[0]\n    threshold = 4 / len(df)\n    n_influential = np.sum(cooks_d > threshold)\n    print(f\"\\n6. INFLUENTIAL OBSERVATIONS\")\n    print(f\"   Number of influential points: {n_influential} ({n_influential/len(df)*100:.1f}%)\")\n    print(f\"   {'✅ Few influential points' if n_influential < len(df)*0.05 else '⚠️ Many influential points'}\")\n    \n    print(f\"\\n{'='*60}\")\n    print(\"OVERALL ASSESSMENT\")\n    print(f\"{'='*60}\")\n    \n    issues = []\n    if shapiro_p < 0.05:\n        issues.append(\"Non-normal residuals\")\n    if bp_p < 0.05:\n        issues.append(\"Heteroscedasticity\")\n    if max_vif > 10:\n        issues.append(\"Multicollinearity\")\n    if n_influential > len(df) * 0.05:\n        issues.append(\"Many influential observations\")\n    \n    if not issues:\n        print(\"✅ All diagnostic checks passed! Model assumptions are satisfied.\")\n    else:\n        print(f\"⚠️ Issues detected: {', '.join(issues)}\")\n        print(\"\\nRecommended actions:\")\n        if \"Non-normal residuals\" in issues:\n            print(\"  - Check for outliers\")\n            print(\"  - Consider transformation of target variable\")\n        if \"Heteroscedasticity\" in issues:\n            print(\"  - Use robust standard errors\")\n            print(\"  - Transform target variable (log, sqrt)\")\n        if \"Multicollinearity\" in issues:\n            print(\"  - Remove or combine correlated features\")\n            print(\"  - Use regularization (Ridge)\")\n        if \"Many influential observations\" in issues:\n            print(\"  - Investigate influential points\")\n            print(\"  - Consider robust regression methods\")\n\n# Run diagnostic report\ndiagnostic_report(model, X, df)

## Summary\n\nIn this notebook, we:\n\n1. ✅ Performed comprehensive residual analysis\n2. ✅ Tested for heteroscedasticity (Breusch-Pagan, White)\n3. ✅ Identified influential observations (Cook's D, leverage)\n4. ✅ Checked for multicollinearity (VIF)\n5. ✅ Tested normality of residuals (Shapiro-Wilk, Jarque-Bera)\n6. ✅ Applied remedial measures (transformations)\n7. ✅ Created automated diagnostic report\n\n## Key Takeaways\n\n- **Always check assumptions** before trusting model results\n- **Visual diagnostics** are as important as statistical tests\n- **Residual plots** reveal patterns that violate assumptions\n- **Influential points** can dramatically affect results\n- **Multicollinearity** makes coefficients unstable\n- **Transformations** can fix many assumption violations\n- **Robust methods** exist when assumptions can't be met\n\n## Diagnostic Workflow\n\n1. **Fit model** and get initial results\n2. **Plot residuals** (vs fitted, Q-Q, histogram)\n3. **Test assumptions** (normality, heteroscedasticity, multicollinearity)\n4. **Identify influential points** (Cook's D, leverage)\n5. **Apply remedies** if needed (transform, robust SE, remove outliers)\n6. **Re-check diagnostics** after remedies\n7. **Report limitations** in final analysis

## Practice Exercises\n\n1. **Create Diagnostic Function**: Build a reusable function for all your models\n2. **Robust Regression**: Implement RANSAC or Huber regression for outlier-resistant models\n3. **Weighted Least Squares**: Apply WLS when heteroscedasticity is present\n4. **Bootstrap Confidence Intervals**: Use bootstrapping for non-normal residuals\n5. **Real Data**: Apply full diagnostic workflow to actual match data\n6. **Automated Remedies**: Create a pipeline that automatically applies appropriate fixes